# Лабораторная работа №5

## Демонстрация RAG-системы с ChromaDB и YandexGPT

Этот ноутбук является основной точкой запуска проекта. Выполняйте ячейки сверху вниз.


In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')
print('PROJECT_ROOT =', PROJECT_ROOT)
print('ENV loaded =', (PROJECT_ROOT / '.env').exists())


PROJECT_ROOT = /Users/exodus/Projects/week5
ENV loaded = True


In [2]:
from src.rag.document_loader import DocumentLoader
from src.rag.chunking import ChunkingStrategy
from src.rag.vector_store import VectorStoreManager
from src.rag.speciality_rag_pipeline import DairyPackagingRAGPipeline
from src.rag.yandex_llm import YandexGPTLLM


In [3]:
DOCUMENTS_DIR = PROJECT_ROOT / 'data' / 'documents'
CHROMA_DIR = PROJECT_ROOT / 'data' / 'chroma_db'
COLLECTION_NAME = 'milk_packaging_rag'

loader = DocumentLoader(str(DOCUMENTS_DIR))
documents = loader.load_directory()
print('Документов загружено:', len(documents))
print(loader.get_statistics())

chunks = ChunkingStrategy.split_documents(
    documents,
    strategy='recursive',
    chunk_size=800,
    chunk_overlap=120,
)
print('Чанков получено:', len(chunks))
print(ChunkingStrategy.get_statistics(chunks))


Пропущен пустой файл: /Users/exodus/Projects/week5/data/documents/sample1.pdf


Документов загружено: 4
{'directory': '/Users/exodus/Projects/week5/data/documents', 'files': {'.docx': 0, '.md': 3, '.pdf': 1, '.txt': 1}, 'total_files': 5, 'total_size_bytes': 4083, 'total_size_mb': 0.0039}
Чанков получено: 5
{'count': 5, 'min_size': 240, 'max_size': 631, 'avg_size': 448.6, 'total_characters': 2243}


In [4]:
vectorstore = VectorStoreManager(
    persist_directory=str(CHROMA_DIR),
    collection_name=COLLECTION_NAME,
    embedding_model='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
)

vectorstore.clear()
index_stats = vectorstore.add_documents(chunks)
print('Индексация завершена:')
print(index_stats)
print(vectorstore.get_statistics())


/Users/exodus/Projects/week5/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Индексация завершена:
{'documents_added': 5, 'collection_name': 'milk_packaging_rag', 'total_documents': 5}
{'collection_name': 'milk_packaging_rag', 'persist_directory': '/Users/exodus/Projects/week5/data/chroma_db', 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'embedding_dimension': 384, 'total_documents': 5}


In [5]:
llm = YandexGPTLLM(
    folder_id=os.environ.get('YANDEX_FOLDER_ID'),
    api_key=os.environ.get('YANDEX_API_KEY'),
    iam_token=os.environ.get('YANDEX_IAM_TOKEN'),
    model_uri=os.environ.get('YANDEX_MODEL_URI'),
    temperature=0.3,
    max_tokens=800,
)

rag = DairyPackagingRAGPipeline(
    vectorstore=vectorstore,
    llm=llm,
    top_k=4,
)

print('RAG pipeline готов к работе')


RAG pipeline готов к работе


In [6]:
question = 'Какие причины дефектов закупорки молочной тары указаны в документах?'
result = rag.query(question)

print('Вопрос:')
print(result['question'])
print('\nОтвет:')
print(result['answer'])
print('\nИсточники:')
for source in result['sources']:
    print('-', source['metadata'].get('file_name'), '| score =', source['similarity_score'])


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Вопрос:
Какие причины дефектов закупорки молочной тары указаны в документах?

Ответ:
- износ механизма укупорки;
- нестабильная подача тары на конвейере;
- загрязнение направляющих;
- неверная настройка прижимного усилия;
- запаздывание сигнала датчика.

Источники:
- system_entities.txt | score = 0.5474
- production_process.md | score = 0.547
- sample2.md | score = 0.5336
- quality_regulations.md | score = 0.4002


In [7]:
question = 'Какие сущности должна хранить информационная система контроля дефектов закупорки?'
result = rag.query(question)

print('Вопрос:')
print(result['question'])
print('\nОтвет:')
print(result['answer'])
print('\nИсточники:')
for source in result['sources']:
    print('-', source['metadata'].get('file_name'), '| score =', source['similarity_score'])


Вопрос:
Какие сущности должна хранить информационная система контроля дефектов закупорки?

Ответ:
Информационная система контроля дефектов закупорки молочной тары должна хранить данные о следующих сущностях:
* дефект (тип отклонения, критичность, время регистрации, статус обработки);
* партия продукции;
* производственная линия;
* укупорочный узел;
* датчики;
* операторы;
* событие контроля;
* уведомление;
* инцидент (связывает дефект с партией, линией, оборудованием и назначенным ответственным сотрудником).

Источники:
- system_entities.txt | score = 0.6963
- quality_regulations.md | score = 0.6093
- production_process.md | score = 0.5981
- production_process.md | score = 0.5619


In [8]:
# Для произвольного запроса меняйте текст ниже и запускайте ячейку.
question = 'Как система должна реагировать на критический рост числа дефектов?'
result = rag.query(question)

print(result['answer'])


При крическом росте числа дефектов система должна:

1. Зарегистрировать время события, идентификатор линии, номер партии, тип дефекта и связанное оборудование.
2. Отправить уведомление оператору.
3. Сохранить инцидент в журнал инцидентов.
